# RoBERTa checkpoint evaluation

Load the saved RoBERTa PCL model from `models/roberta_pcl_imbalance_final_checkpoint-944_*`, run evaluation on dev, and write predictions to `dev.txt` and `test.txt` at project root.

In [8]:
import re
import html
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

MAX_LENGTH = 512
EVAL_BATCH_SIZE = 32

# Checkpoint dir: default is 0.558; set to roberta_pcl_imbalance_final_checkpoint-944_0608.040 if you have that copy
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if (cwd.parent / "data").exists() else cwd
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "roberta_pcl_imbalance_final_checkpoint-944_0608.040"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint:   {CHECKPOINT_DIR}")
print(f"Device:       {device}")

Project root: /vol/bitbucket/tq422/NLP_cw
Checkpoint:   /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_imbalance_final_checkpoint-944_0608.040
Device:       cuda


In [9]:
tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(CHECKPOINT_DIR), num_labels=2)
model.to(device)
print("Model and tokenizer loaded.")

Model and tokenizer loaded.


## Load and preprocess dev & test data

In [10]:
def strip_html_and_clean(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return text.strip() if isinstance(text, str) else ""
    no_tags = re.sub(r"<[^>]+>", "", text)
    unescaped = html.unescape(no_tags)
    return re.sub(r"\s+", " ", unescaped).strip()

# Full data and splits
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t", skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)
df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)
df_sub = df[["par_id", "text", "label_binary"]].drop_duplicates(subset=["par_id"], keep="first").copy()
df_sub["text"] = df_sub["text"].fillna("").astype(str)

# Dev set
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")
dev_df = a_dev[["par_id"]].merge(df_sub, on="par_id", how="left")
dev_df["text"] = dev_df["text"].fillna("").astype(str).apply(strip_html_and_clean).replace("", " ")
dev_df["label_binary"] = dev_df["label_binary"].fillna(0).astype(int)

# Test set
test_df = pd.read_csv(
    RAW_DATA_DIR / "task4_test.tsv",
    sep="\t",
    names=["id", "par_id", "keyword", "country_code", "text"],
)
test_df["text"] = test_df["text"].fillna("").astype(str).apply(strip_html_and_clean).replace("", " ")

print(f"Dev:  {len(dev_df)} samples")
print(f"Test: {len(test_df)} samples")

Dev:  2094 samples
Test: 3832 samples


In [11]:
class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
test_dataset = PCLDataset(
    test_df["text"],
    np.zeros(len(test_df), dtype=int),
    tokenizer,
    max_length=MAX_LENGTH,
)

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = (preds == labels).mean()
    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(PROJECT_ROOT / "output" / "eval_tmp"),
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
    ),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Evaluate on dev and write dev.txt

In [13]:
eval_metrics = trainer.evaluate(dev_dataset)
print("Dev metrics:")
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

pred_output = trainer.predict(dev_dataset)
dev_preds = np.argmax(pred_output.predictions, axis=-1)
dev_labels = pred_output.label_ids

print("\nClassification report (dev):")
print(classification_report(dev_labels, dev_preds, target_names=["No PCL", "PCL"], digits=4, zero_division=0))

dev_path = PROJECT_ROOT / "dev.txt"
with open(dev_path, "w", encoding="utf-8") as f:
    for p in dev_preds:
        f.write(f"{p}\n")
print(f"\nSaved dev predictions to {dev_path}")

Dev metrics:
  eval_loss: 0.3557
  eval_model_preparation_time: 0.0058
  eval_accuracy: 0.9255
  eval_precision_pos: 0.6080
  eval_recall_pos: 0.6080
  eval_f1_pos: 0.6080
  eval_f1_macro: 0.7834
  eval_runtime: 6.9215
  eval_samples_per_second: 302.5360
  eval_steps_per_second: 9.5360

Classification report (dev):
              precision    recall  f1-score   support

      No PCL     0.9588    0.9588    0.9588      1895
         PCL     0.6080    0.6080    0.6080       199

    accuracy                         0.9255      2094
   macro avg     0.7834    0.7834    0.7834      2094
weighted avg     0.9255    0.9255    0.9255      2094


Saved dev predictions to /vol/bitbucket/tq422/NLP_cw/dev.txt


## Predict on test and write test.txt

In [14]:
test_pred_output = trainer.predict(test_dataset)
test_preds = np.argmax(test_pred_output.predictions, axis=-1)

test_path = PROJECT_ROOT / "test.txt"
with open(test_path, "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to {test_path}")

Saved test predictions to /vol/bitbucket/tq422/NLP_cw/test.txt
